# Qwen3-VL ハンズオン: 画像 + テキスト入力をまず動かす

このNotebookは、**Hugging Faceで公開されているQwen系VLMをローカルで動かす最小例**を、手を動かしながら理解できる形に整理したものです。

> メモ: ユーザー依頼では `Qwen3.5` と書かれていましたが、**画像+テキスト入力のVLM系列は Hugging Face 上では `Qwen3-VL-*`** として公開されています。通常の `Qwen3.5-*` は主にテキストLLMです。そこで本Notebookでは、VLMとして `Qwen/Qwen3-VL-2B-Instruct` を使います。

## 0. 何をやるか

このNotebookで行うこと:
1. 実行環境を確認する
2. サンプル画像を用意する
3. `AutoModelForImageTextToText` と `AutoProcessor` を読み込む
4. 画像 + テキストのメッセージを作る
5. 推論を実行して、出力を読む

まずは**確実に一度動かす**ことを優先しています。

In [ ]:
from pathlib import Path
import requests
from PIL import Image
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
IMAGE_PATH = Path("../data/demo.jpeg")
IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)

print({
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "model": MODEL_ID,
})

## 1. サンプル画像を取得して表示

今回はQwen公式サンプル画像を使います。まずは画像を見て、後でモデルの回答が妥当か確認できるようにします。

In [ ]:
if not IMAGE_PATH.exists():
    response = requests.get(IMAGE_URL, timeout=60)
    response.raise_for_status()
    IMAGE_PATH.write_bytes(response.content)

image = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis("off")
plt.show()
print(IMAGE_PATH)

## 2. モデルとProcessorをロード

- `AutoModelForImageTextToText`: 画像+テキストを受けるVLM本体
- `AutoProcessor`: テキスト整形、画像前処理、chat template適用をまとめて担当

GPUが使える場合は `bfloat16`、そうでなければCPU向けに `float32` を使います。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print({"device": device, "dtype": str(dtype)})

## 3. 画像 + テキストの入力を作る

Qwen3-VLは chat template で入力を作るのが分かりやすいです。
`content` の中に `image` と `text` を並べると、**マルチモーダル入力**になります。

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this image in 5 concise bullet points."},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

{k: tuple(v.shape) if hasattr(v, "shape") else type(v) for k, v in inputs.items()}

## 4. 推論

`generate()` で応答を作ります。
出力には**入力トークンも含まれる**ので、後ろの新規生成部分だけを切り出して decode します。

In [ ]:
with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

prompt_len = inputs["input_ids"].shape[1]
trimmed = generated_ids[:, prompt_len:]
output_text = processor.batch_decode(
    trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print(output_text)

## 5. ここまでの理解

最低限押さえるポイント:

- **モデル本体**: `AutoModelForImageTextToText`
- **前処理**: `AutoProcessor`
- **マルチモーダル入力**: `content` に `image` と `text` を混在させる
- **推論**: `processor.apply_chat_template(...)` → `model.generate(...)`

この形がまずの基本です。ここから、複数画像、動画、解像度制御、量子化、KV cache などに発展できます。